# IRWA Project - Part 2: Indexing and Evaluation

## Previous
This section reuses the data loading, Product objects, text preprocessing, attribute cleaning and numeric metadata from Part 1. Both documents and queries use the same processing. The following sections match the original assignment: Part1-1 to Part1-3 and Part2-1 to Part2-3.

Run the cells in order from this folder or the repository root. The NLTK English stopword list must be available; if needed, run `nltk.download("stopwords")` once.

### Loading the corpus

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from IPython.display import display

project_root = next(
    directory for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "data/fashion_products_dataset.json").is_file()
)
with (project_root / "data/fashion_products_dataset.json").open(encoding="utf-8") as fp:
    raw_products = json.load(fp)

In [2]:
class Product:
  def __init__(self, _id, actual_price, average_rating, brand, category, crawled_at, description, discount, images, out_of_stock, pid, product_details, seller, selling_price, sub_category, title, url):
    self._id = _id
    self.actual_price = actual_price
    self.average_rating = average_rating
    self.brand = brand
    self.category = category
    self.crawled_at = crawled_at
    self.description = description
    self.discount = discount
    self.images = images
    self.out_of_stock = out_of_stock
    self.pid = pid
    self.product_details = product_details
    self.seller = seller
    self.selling_price = selling_price
    self.sub_category = sub_category
    self.title = title
    self.url = url

products = [Product(**record) for record in raw_products]
products_by_pid = {product.pid: product for product in products} 

### Text preprocessing and attribute cleaning
We reuse lowercasing, splitting joined words, normalising `t-shirt`, stopword removal and Porter stemming. Selected product attributes are cleaned before indexing.

In [3]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def stem_term(word):
    return stemmer.stem(word)

In [4]:
split_chars = '/.,-'

def split_joined_words(text):
  chars = list(text)
  for i, ch in enumerate(text):
    if ch in split_chars:
      prev_is_letter = i > 0 and text[i - 1].isalpha()
      next_is_letter = i < len(text) - 1 and text[i + 1].isalpha()
      if prev_is_letter or next_is_letter:
        chars[i] = ' '
  return ''.join(chars)

def merge_tshirt(words):
    result = []
    i = 0
    while i < len(words):
        if words[i] == 't' and i + 1 < len(words) and words[i + 1] in ('shirt', 'shirts'):
            result.append('tshirt')
            i += 2
        else:
            result.append('tshirt' if words[i] == 'tshirts' else words[i])
            i += 1
    return result

def build_terms(text: str):
    text = split_joined_words(text.lower())
    words = merge_tshirt(text.split())                                                    
    words = [w for w in words if w not in stop_words]
    words = [''.join(ch for ch in w if ch.isalnum() or ch in '/%.') for w in words]
    words = [w.strip('./') for w in words]
    words = [w for w in words if w and w not in stop_words]
    return [stem_term(w) for w in words]

In [5]:
def preprocess_details(details):
    keep = {
        'fabric', 'pattern', 'sleeve', 'sleeve type', 'type', 'generic name',
        'ideal for', 'fit', 'brand fit', 'color', 'brand color', 'secondary color',
        'shade', 'neck type', 'neck', 'collar', 'occasion', 'closure', 'fly',
        'top closure', 'hooded', 'reversible', 'stretchable', 'rise', 'hem',
        'faded', 'distressed', 'length type', 'pocket type', 'pockets',
        'waistband', 'character', 'fabric details', 'knit type', 'weave type',
        'material', 'outer material', 'sole material', 'lining', 'lining material',
        'top fabric', 'bottom fabric', 'top type', 'bottom type', 'coat type',
        'upper pattern',
    }
    boolean_fields = {'hooded', 'reversible', 'stretchable'}
    missing_values = {
        '', 'na', 'n/a', 'n.a.', 'none', 'null', 'nan', '-', '--',
        'not available', 'not applicable',
    }
    fit_aliases = {
        'slim fit': 'slim',
        'regular fit': 'regular',
        'skinny fit': 'skinny',
        'loose fit': 'loose',
        'relaxed fit': 'relaxed',
        'comfort fit': 'comfort',
        'tailored fit': 'tailored',
        'super slim fit': 'super slim',
    }
    product_values = []
    seen_values = set()
    for detail in details or []:
        for key, value in detail.items():
            key = ' '.join(str(key).lower().split())
            if key not in keep or value is None:
                continue
            for part in str(value).split(','):
                part = ' '.join(part.lower().split())
                if part in missing_values:
                    continue
                if key in boolean_fields:
                    if part != 'yes':
                        continue
                    part = key
                elif part.isdigit() or part == 'no' or part.startswith(('no ', 'not ')):
                    continue
                elif key in {'fit', 'brand fit'}:
                    part = fit_aliases.get(part, part)
                if part not in seen_values:
                    seen_values.add(part)
                    product_values.append(part)
    return product_values

### Numeric metadata
We keep `quality_score = rating / 5`, replacing missing ratings with the observed mean. Prices, discounts and stock remain separate from text. The ranking in this assignment uses TF-IDF; combining it with quality or commercial criteria is left for a later evaluation.

In [6]:
import math

def parse_numeric(value):
    if value is None:
        return None
    cleaned = str(value).replace(",", "").replace("% off", "").strip()
    try:
        number = float(cleaned)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None

numeric_field_names = ("average_rating", "selling_price", "actual_price", "discount")
numeric_fields_by_pid = {
    product.pid: {
        field: parse_numeric(getattr(product, field))
        for field in numeric_field_names
    }
    for product in products
}
available_ratings = [metadata["average_rating"]
                     for metadata in numeric_fields_by_pid.values()
                     if metadata["average_rating"] is not None]
mean_rating = sum(available_ratings) / len(available_ratings) if available_ratings else None
for product in products:
    metadata = numeric_fields_by_pid[product.pid]
    rating = metadata["average_rating"]
    rating_for_quality = rating if rating is not None else mean_rating
    metadata["quality_score"] = rating_for_quality / 5 if rating_for_quality is not None else None
    metadata["out_of_stock"] = product.out_of_stock

## Part1-1 - Inverted index and AND retrieval
We reuse the positional index functions from Part 1 to build separate indexes for title/description and selected attributes. Each index stores `term -> [[pid, [positions]], ...]`. Positions refer to processed tokens within each field.

In [7]:
def get_product_index(terms, pid):
  index = {}

  for position, term in enumerate(terms):
      try:
          index[term][1].append(position)
      except KeyError:
          index[term] = [pid, [position]]
  return index

In [8]:
def create_indexes(products: list[Product]):
  text_index = defaultdict(list)
  detail_index = defaultdict(list)

  for product in products:

    text_terms = build_terms(product.title + " " + product.description)

    for term, term_info in get_product_index(text_terms, product.pid).items():
      text_index[term].append(term_info)


    current_detail_index = preprocess_details(product.product_details)
    detail_terms = build_terms(' '.join(current_detail_index))

    for term, term_info in get_product_index(detail_terms, product.pid).items():
      detail_index[term].append(term_info)

  return text_index, detail_index

In [9]:
text_index, detail_index = create_indexes(products)
print(f"Title and description terms: {len(text_index):,}")
print(f"Attribute terms: {len(detail_index):,}")

Title and description terms: 5,288
Attribute terms: 1,381


### Conjunctive retrieval
For each term, we take the union of the products found in either index. We then intersect the sets for all query terms: every term must occur in the same product. An empty query or a query containing an unknown term returns an empty list.

In [10]:
def search_and(query):
    terms = list(dict.fromkeys(build_terms(query)))
    if not terms:
        return []
    matches = []
    for term in terms:
        term_pids = {pid for pid, positions in text_index.get(term, [])}
        term_pids.update(pid for pid, positions in detail_index.get(term, []))
        matches.append(term_pids)
    return sorted(set.intersection(*matches))

## Part1-2 - Five test queries
Document frequency is inspected to select terms that occur in the corpus. The five queries combine clothing types and specific attributes, covering gender, fabric, colour and fit. The initial examples are AND matches, before relevance ranking.

In [11]:
vocabulary = set(text_index) | set(detail_index)
document_frequency = {
    term: len({pid for pid, positions in text_index.get(term, [])}
              | {pid for pid, positions in detail_index.get(term, [])})
    for term in vocabulary
}
display(pd.Series(document_frequency, name="products")
        .sort_values(ascending=False).head(15).to_frame())

test_queries = {
    1: "women cotton tshirt",
    2: "men slim jeans blue",
    3: "women printed kurta",
    4: "men cotton track pants",
    5: "women full sleeve sweatshirt",
}
query_summary = pd.DataFrame([
    {"query_id": query_id, "query": query,
     "terms": " ".join(build_terms(query)), "matches": len(search_and(query))}
    for query_id, query in test_queries.items()
])
display(query_summary)

,products
cotton,19590
sleev,17904
neck,14569
women,13699
blend,13593
men,13391
regular,13087
tshirt,12899
solid,11920
round,9479


,query_id,query,terms,matches
0,1,women cotton tshirt,women cotton tshirt,5088
1,2,men slim jeans blue,men slim jean blue,219
2,3,women printed kurta,women print kurta,40
3,4,men cotton track pants,men cotton track pant,437
4,5,women full sleeve sweatshirt,women full sleev sweatshirt,655


## Part1-3 - TF-IDF ranking
Term frequencies in title/description and attributes are added together while the original indexes remain separate. Both fields contribute equally, without additional weights.

We use `TF = 1 + log(frequency)` and `IDF = log(N / DF) + 1`. TF-IDF vectors are computed for the full document and the query, and results are ranked by cosine similarity. Normalisation uses the entire document vocabulary to account for length. Only candidates satisfying AND are ranked; ties are resolved by `pid`.

In [12]:
term_counts = defaultdict(Counter)
for index in (text_index, detail_index):
    for term, postings in index.items():
        for pid, positions in postings:
            term_counts[term][pid] += len(positions)

idf = {term: math.log(len(products) / len(counts)) + 1
       for term, counts in term_counts.items()}
tfidf = {}
document_norms = defaultdict(float)
for term, counts in term_counts.items():
    tfidf[term] = {}
    for pid, count in counts.items():
        weight = (1 + math.log(count)) * idf[term]
        tfidf[term][pid] = weight
        document_norms[pid] += weight ** 2
document_norms = {pid: math.sqrt(value) for pid, value in document_norms.items()}

def search_tfidf(query):
    candidates = search_and(query)
    if not candidates:
        return []
    query_counts = Counter(build_terms(query))
    query_weights = {term: (1 + math.log(count)) * idf[term]
                     for term, count in query_counts.items()}
    query_norm = math.sqrt(sum(weight ** 2 for weight in query_weights.values()))
    results = []
    for pid in candidates:
        dot_product = sum(weight * tfidf[term][pid]
                          for term, weight in query_weights.items())
        score = dot_product / (document_norms[pid] * query_norm)
        results.append((pid, score))
    return sorted(results, key=lambda result: (-result[1], result[0]))

In [13]:
for query_id, query in test_queries.items():
    print(f"{query_id}: {query}")
    display(pd.DataFrame([
        {"pid": pid, "title": products_by_pid[pid].title, "score": score}
        for pid, score in search_tfidf(query)[:5]
    ]))

1: women cotton tshirt


,pid,title,score
0,TSHFGZKPMGYWN4CH,Printed Women Round Neck Black T-Shirt,0.529038
1,TSHFVXGQ73ZRG9AW,Printed Women Round Neck Blue T-Shirt,0.496607
2,TSHFZZ4YSYAUJ3Q9,Solid Women Round Neck Black T-Shirt,0.488823
3,TSHFGZKNBFA8HKYE,Printed Women Round Neck Red T-Shirt,0.481508
4,TSHFGZKPHARMSFY9,Printed Women Round Neck Red T-Shirt,0.481508


2: men slim jeans blue


,pid,title,score
0,JEAFTGSGTYKZGAEZ,Slim Men Blue Jeans,0.694192
1,JEAFSKYHZHSZZC9S,Slim Men Blue Jeans,0.685251
2,JEAFV8WURBM7PGYT,Slim Men Blue Jeans,0.663528
3,JEAFVDRJZ5RWAFHB,Slim Men Blue Jeans,0.663528
4,JEAFQF6JBUSEXHVF,Slim Men Blue Jeans,0.640315


3: women printed kurta


,pid,title,score
0,FABFWN82HVT5KAH2,Cotton Printed Kurta Fabric (Unstitched),0.658882
1,ETHFT9HQRGEXHETK,Women Kurta and Pyjama Set Cotton Blend,0.506728
2,ETHFT9HQDDYTZZT5,Women Kurta and Pyjama Set Cotton Blend,0.476166
3,ETHFT9HQNKWEF2EH,Women Kurta and Pyjama Set Cotton Blend,0.476166
4,ETHFY7FBXY6ZDTKN,Women Kurta and Pyjama Set Pure Cotton,0.457426


4: men cotton track pants


,pid,title,score
0,TKPFRR8388WSM4ZR,Solid Men Blue Track Pants,0.851043
1,TKPFRR83HDEPCQD6,Solid Men Blue Track Pants,0.851043
2,TKPFV5KFPXFRFXG4,Solid Men Blue Track Pants,0.851043
3,TKPFJQBGEZZWPUET,Solid Men Black Track Pants,0.839363
4,TKPFJQEMYRQUV8XD,Solid Men Black Track Pants,0.839363


5: women full sleeve sweatshirt


,pid,title,score
0,SWSFZVTTQCB4SJ7F,Full Sleeve Solid Women Sweatshirt,0.846651
1,SWSFZVTNGM4HG8BC,Full Sleeve Printed Women Sweatshirt,0.826364
2,SWSFQGS456JAZCQB,Full Sleeve Printed Women Sweatshirt,0.812219
3,SWSFV5JNFW5HZQWY,Full Sleeve Printed Women Sweatshirt,0.812219
4,SWSFZAUYZFAGEPKK,Full Sleeve Printed Women Sweatshirt,0.797377


## Part2-1 - Evaluation metrics
We follow the lab structure: sort by score with NumPy, obtain the ordered labels and calculate metrics for each query. Ties preserve the ranking order.

- **P@K:** relevant results retrieved divided by K. If fewer than K products are retrieved, empty positions contribute no hits.
- **R@K:** relevant results retrieved divided by all known relevant products for the query.
- **AP@K:** sum of precisions at relevant positions, divided by the number of relevant products retrieved in the first K, as in the lab. Relevant products that are not retrieved therefore do not lower AP@K; R@K captures them instead.
- **F1@K:** harmonic mean of P@K and R@K.
- **MAP@K and MRR@K:** mean AP@K and mean reciprocal rank of the first relevant result, respectively, across queries.
- **NDCG@K:** DCG with gain `2**rel - 1` and discount `log2(position + 1)`, divided by the ideal DCG of all known labels.

A value of zero is returned when there are no known relevant products or the ideal DCG is zero. Intermediate calculations are not rounded.

In [14]:
def ranked_labels(y_true, y_score, k):
    order = np.argsort(-np.asarray(y_score), kind="stable")
    return np.asarray(y_true)[order][:k]

def precision_at_k(y_true, y_score, k=10):
    labels = ranked_labels(y_true, y_score, k)
    return float(np.sum(labels) / k)

def recall_at_k(y_true, y_score, total_relevant, k=10):
    if total_relevant == 0:
        return 0.0
    return float(np.sum(ranked_labels(y_true, y_score, k)) / total_relevant)

def avg_precision_at_k(y_true, y_score, k=10):
    labels = ranked_labels(y_true, y_score, k)
    hits = np.sum(labels)
    if hits == 0:
        return 0.0
    precisions = np.cumsum(labels) / np.arange(1, len(labels) + 1)
    return float(np.sum(precisions * labels) / hits)

def f1_at_k(y_true, y_score, total_relevant, k=10):
    precision = precision_at_k(y_true, y_score, k)
    recall = recall_at_k(y_true, y_score, total_relevant, k)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def rr_at_k(y_true, y_score, k=10):
    labels = ranked_labels(y_true, y_score, k)
    return 1.0 / (np.argmax(labels) + 1) if np.any(labels) else 0.0

def dcg_at_k(y_true, y_score, k=10):
    labels = ranked_labels(y_true, y_score, k)
    return float(np.sum((2.0 ** labels - 1) / np.log2(np.arange(len(labels)) + 2)))

def ndcg_at_k(y_true, y_score, ideal_labels, k=10):
    ideal_dcg = dcg_at_k(ideal_labels, ideal_labels, k)
    return dcg_at_k(y_true, y_score, k) / ideal_dcg if ideal_dcg else 0.0

def map_at_k(evaluations, k=10):
    return float(np.mean([
        avg_precision_at_k(labels, scores, k)
        for labels, scores, total_relevant, ideal_labels in evaluations
    ]))

def mrr_at_k(evaluations, k=10):
    return float(np.mean([
        rr_at_k(labels, scores, k)
        for labels, scores, total_relevant, ideal_labels in evaluations
    ]))

### Validation protocol
The official queries are `women full sleeve sweatshirt cotton` (1) and `men slim jeans blue` (2). `validation_labels.csv` contains binary judgments for a subset of the corpus, rather than every product.

AND + TF-IDF is run on the full corpus, then only results judged for the query are retained in the same order. Labelled products that do not satisfy AND are not added to the ranking, but their labels still count towards the total number of relevant products (used by R@K) and the ideal ranking. Unjudged products are not treated as irrelevant. The results describe this judged set and do not by themselves estimate performance on the full corpus.

We calculate K = 3, 5 and 10. The per-query rows contain P@K, R@K, AP@K, F1@K, RR@K and NDCG@K; a second table contains MAP@K and MRR@K for the two queries. The following results section displays only numeric tables with three decimal places.

In [15]:
validation = pd.read_csv(project_root / "data/validation_labels.csv")
official_queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue",
}
evaluations = []
for query_id, query in official_queries.items():
    judgments = validation.loc[validation["query_id"] == query_id].set_index("pid")["labels"]
    judged_ranking = [(pid, score) for pid, score in search_tfidf(query)
                      if pid in judgments.index]
    labels = np.array([judgments[pid] for pid, score in judged_ranking], dtype=int)
    scores = np.array([score for pid, score in judged_ranking])
    evaluations.append((labels, scores, int(judgments.sum()), judgments.to_numpy()))

metric_rows = []
mean_rows = []
for k in (3, 5, 10):
    for query_id, (labels, scores, total_relevant, ideal_labels) in zip(official_queries, evaluations):
        metric_rows.append({
            "query_id": query_id, "K": k,
            "P@K": precision_at_k(labels, scores, k),
            "R@K": recall_at_k(labels, scores, total_relevant, k),
            "AP@K": avg_precision_at_k(labels, scores, k),
            "F1@K": f1_at_k(labels, scores, total_relevant, k),
            "RR@K": rr_at_k(labels, scores, k),
            "NDCG@K": ndcg_at_k(labels, scores, ideal_labels, k),
        })
    mean_rows.append({"K": k, "MAP@K": map_at_k(evaluations, k),
                      "MRR@K": mrr_at_k(evaluations, k)})
metrics = pd.DataFrame(metric_rows).set_index(["query_id", "K"]).sort_index()
mean_metrics = pd.DataFrame(mean_rows).set_index("K")

## Part2-2 - Official validation results

In [16]:
display(metrics.style.format("{:.3f}"))
display(mean_metrics.style.format("{:.3f}"))

,MAP@K,MRR@K
K,,
3,1.000,1.000
5,1.000,1.000
10,0.974,1.000


## Part2-3 - Own relevance judgments and analysis
### Part2-3.a - Binary relevance judgments
A binary label is assigned to **every product for each of the five queries**. Relevance represents a shopping need, rather than the mere presence of every query word. Original fields and explicit rules are used, independently of TF-IDF scores and index preprocessing.

| Query | Relevant if all of these criteria are met |
| --- | --- |
| women cotton tshirt | Women's T-shirt with cotton in the Fabric field; blends and polycotton are accepted. |
| men slim jeans blue | Men's blue jeans, including light or dark blue, with a slim fit; skinny or tapered alone is insufficient. |
| women printed kurta | Women's printed kurta; sets containing a kurta are accepted, but unstitched fabric is excluded. |
| men cotton track pants | Men's track pants with cotton in Fabric; blends are accepted. |
| women full sleeve sweatshirt | Women's sweatshirt with full-length sleeves. |

For gender, Ideal For takes priority, with the title as a fallback. For fabric, fit, colour, pattern and sleeve length, the corresponding attribute takes priority; the title is a fallback except for fabric. The description does not determine relevance because it may contain advertising or references to other garments. If a required attribute cannot be confirmed using these priorities, the label is 0. Contradictions between fields are a limitation of these rules. Garment type is taken from the title; unstitched fabric is explicitly excluded.

These are **automatic labels based on explicit judgment criteria**, reviewed through examples; they are not individual human annotations of every pair. They provide a reproducible operational reference and should be validated by human judges before being used as definitive ground truth. Official labels are neither reused nor modified, including for the matching jeans query.

In [17]:
import re

def judgment_text(value):
    return " ".join(re.findall(r"[a-z0-9]+", str(value).lower()))

def product_judgments(product):
    title = judgment_text(product.title)
    details = {}
    for detail in product.product_details or []:
        for key, value in detail.items():
            text = judgment_text(value) if value is not None else ""
            if text not in {"", "na", "n a", "none", "null", "not applicable"}:
                details.setdefault(judgment_text(key), []).append(text)

    def attribute(keys, fallback=True):
        values = [value for key in keys for value in details.get(key, [])]
        return " ".join(values) if values else (title if fallback else "")

    gender = attribute(["ideal for"])
    women = bool(re.search(r"\bwomen\b", gender))
    men = bool(re.search(r"\bmen\b", gender))
    cotton = "cotton" in attribute(["fabric"], fallback=False)
    fit = attribute(["fit", "brand fit"])
    slim = bool(re.search(r"\bslim\b|\bslimfit\b", fit))
    blue = bool(re.search(r"\bblue\b", attribute(["color", "brand color", "shade"])))
    printed = bool(re.search(r"\bprinted\b|\bprint\b", attribute(["pattern"])))
    full_sleeve = bool(re.search(r"\bfull sleeves?\b|\blong sleeves?\b",
                                 attribute(["sleeve", "sleeve type"])))
    fabric_only = bool(re.search(r"\bfabric\b|\bunstitched\b", title))
    tshirt = bool(re.search(r"\bt shirts?\b|\btshirts?\b", title))
    jeans = bool(re.search(r"\bjeans\b", title))
    kurta = bool(re.search(r"\bkurtas?\b", title)) and not fabric_only
    track_pants = bool(re.search(r"\btrack pants?\b|\btrackpants?\b", title))
    sweatshirt = bool(re.search(r"\bsweatshirts?\b", title))
    return [int(women and cotton and tshirt),
            int(men and slim and jeans and blue),
            int(women and printed and kurta),
            int(men and cotton and track_pants),
            int(women and full_sleeve and sweatshirt)]

own_labels = pd.DataFrame([
    {"query_id": query_id, "pid": product.pid, "label": label}
    for product in products
    for query_id, label in enumerate(product_judgments(product), start=1)
])
output_folder = project_root / "project_progress/part_2"
own_labels.to_csv(output_folder / "own_relevance_labels.csv", index=False)
display(own_labels.groupby("query_id")["label"].agg(["count", "sum"])
        .rename(columns={"count": "labeled_products", "sum": "relevant_products"}))

,labeled_products,relevant_products
query_id,,
1,28080,5512
2,28080,115
3,28080,38
4,28080,390
5,28080,613


### Part2-3.b - Evaluation results and analysis
The five queries are evaluated on the full corpus without filtering the ranking to a sample. Recall and IDCG use all positives in the own labels. The metric functions from Part2-1 are reused. Examples show the first ten results for each query, including their attributes and labels, to make the decisions reviewable.

In [18]:
own_evaluations = []
review_rows = []
for query_id, query in test_queries.items():
    judgments = own_labels.loc[own_labels["query_id"] == query_id].set_index("pid")["label"]
    ranking = search_tfidf(query)
    labels = np.array([judgments[pid] for pid, score in ranking], dtype=int)
    scores = np.array([score for pid, score in ranking])
    own_evaluations.append((labels, scores, int(judgments.sum()), judgments.to_numpy()))
    for rank, (pid, score) in enumerate(ranking[:10], start=1):
        product = products_by_pid[pid]
        review_rows.append({"query_id": query_id, "rank": rank, "pid": pid,
                            "title": product.title, "label": int(judgments[pid]),
                            "details": product.product_details})

own_rows = []
own_mean_rows = []
for k in (3, 5, 10):
    for query_id, (labels, scores, total_relevant, ideal_labels) in zip(test_queries, own_evaluations):
        own_rows.append({
            "query_id": query_id, "K": k,
            "P@K": precision_at_k(labels, scores, k),
            "R@K": recall_at_k(labels, scores, total_relevant, k),
            "AP@K": avg_precision_at_k(labels, scores, k),
            "F1@K": f1_at_k(labels, scores, total_relevant, k),
            "RR@K": rr_at_k(labels, scores, k),
            "NDCG@K": ndcg_at_k(labels, scores, ideal_labels, k),
        })
    own_mean_rows.append({"K": k, "MAP@K": map_at_k(own_evaluations, k),
                         "MRR@K": mrr_at_k(own_evaluations, k)})
own_metrics = pd.DataFrame(own_rows).set_index(["query_id", "K"]).sort_index()
own_mean_metrics = pd.DataFrame(own_mean_rows).set_index("K")
display(own_metrics.style.format("{:.3f}"))
display(own_mean_metrics.style.format("{:.3f}"))
review_examples = pd.DataFrame(review_rows)
review_examples.to_csv(output_folder / "own_review_examples.csv", index=False)
with pd.option_context("display.max_colwidth", 140):
    display(review_examples)

,MAP@K,MRR@K
K,,
3,0.917,0.900
5,0.936,0.900
10,0.957,0.900


,query_id,rank,pid,title,label,details
0,1,1,TSHFGZKPMGYWN4CH,Printed Women Round Neck Black T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Short Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Blend'}, {'Sales Package': '1 Tshirt'}, {'Pa..."
1,1,2,TSHFVXGQ73ZRG9AW,Printed Women Round Neck Blue T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Half Sleeve'}, {'Fit': 'Slim'}, {'Fabric': 'Pure Cotton'}, {'Sales Package': 'Pack of 1 T-shirt'}, ..."
2,1,3,TSHFZZ4YSYAUJ3Q9,Solid Women Round Neck Black T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Full Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Blend'}, {'Sales Package': '1'}, {'Pack of': ..."
3,1,4,TSHFGZKNBFA8HKYE,Printed Women Round Neck Red T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Short Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Blend'}, {'Sales Package': '1 Tshirt'}, {'Pa..."
4,1,5,TSHFGZKPHARMSFY9,Printed Women Round Neck Red T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Short Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Blend'}, {'Sales Package': '1 Tshirt'}, {'Pa..."
5,1,6,TSHFUNN3NNHHH9RQ,Printed Women Round Neck White T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Short Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Pure Cotton'}, {'Pack of': '1'}, {'Style Code': 'SB..."
6,1,7,TSHFTN3UMHSCYTUH,Printed Women Round Neck Blue T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Half Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Blend'}, {'Pack of': '1'}, {'Style Code': 'V-..."
7,1,8,TSHFZVT7YJZF7GYY,Printed Women Round Neck Black T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Short Sleeve'}, {'Fit': 'Slim'}, {'Fabric': 'Pure Cotton'}, {'Pack of': '1'}, {'Style Code': '10879..."
8,1,9,TSHFYNF5PPDHWEYE,Printed Women Round Neck White T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Short Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Blend'}, {'Sales Package': '1 t-shirt'}, {'P..."
9,1,10,TSHFNYTT4WNPPPVQ,Solid Women Round Neck Black T-Shirt,1,"[{'Type': 'Round Neck'}, {'Sleeve': 'Full Sleeve'}, {'Fit': 'Regular'}, {'Fabric': 'Cotton Blend'}, {'Sales Package': '1'}, {'Pack of': ..."


#### What each metric contributes

- **P@K** indicates how many of the first K results satisfy the criteria. It does not distinguish between hits at the beginning or the end of the cutoff.
- **R@K** measures the fraction of known relevant products appearing in the first K results. A query with thousands of positives can have high P@10 and low R@10 because only ten products are inspected.
- **AP@K** combines hits and position through cumulative precision at each hit. Its denominator is the number of relevant products retrieved in the first K, so it can equal 1 even when overall recall is low, and it ignores relevant products that were missed.
- **F1@K** balances precision and recall. It will be low when the number of relevant products is large relative to K, even if the initial ranking is useful.
- **MAP@K** summarises AP across the five queries, giving each equal weight. It can hide poor performance on an individual query and should be read alongside the per-query table.
- **MRR@K** focuses on the first hit. It can be high even if the remaining results are poor, because later hits are not rewarded.
- **NDCG@K** rewards hits at high ranks relative to the ideal ranking. It supports graded relevance, although binary labels are used here. Unlike AP, it applies logarithmic position discounts.

The official and own results use different evaluation sets and criteria: official validation filters to judged products, while the own evaluation uses labels for the full corpus. They should not be compared as an improvement or decline in the same experiment.

#### Analysis of the observed results

At K = 10, queries 1, 2, 4 and 5 have P@10 = 1.000 under the defined criteria. Query 3 has P@10 = 0.900, AP@10 = 0.786 and NDCG@10 = 0.780: its first result, `FABFWN82HVT5KAH2`, is **unstitched kurta fabric**, rather than a finished garment. The first hit is at position two, giving RR = 0.500. This case shows that satisfying AND and obtaining a high score does not guarantee satisfying the shopping need.

MAP increases from 0.917 at K = 3 to 0.957 at K = 10 because later hits for query 3 partially offset its first error. MRR remains 0.900 at all three cutoffs: increasing K does not change the position of the first hit for any query.

Low recall in the first ten positions has two distinct causes. Query 1 has 5,512 products labelled relevant, so ten hits yield only R@10 = 0.002; even an ideal ranking would have the same limit at K = 10. In addition, AND retrieves only 4,779 of these 5,512 positives, showing coverage losses before ranking. For example, `TSHFWTPXPEDHXUQQ` is a women's T-shirt with Fabric = Polycotton, which the criteria accept as cotton, but the index does not map polycotton to cotton.

For query 2, 114 of 115 positives are retrieved. The missed product `JEAEKNV83HZRKETB` has Fit = Slimfit and the title Regular Men Blue Jeans: the rule prioritises Slimfit, but the index does not convert it to slim. This is also an example of inconsistent corpus data that deserves human review. For query 4, 330 of 390 positives are retrieved; `TKPEYQ6YSFS8KEHK` contains Polycotton and illustrates the same vocabulary mismatch. Queries 3 and 5 retrieve all their positives under these rules, although this does not mean all appear in the first ten positions.

These values describe agreement with the annotation rules, rather than an unbiased estimate of human satisfaction. The criteria use some of the same fields as the index, which may favour agreement. Reviewing the examples confirms the fabric false positive and the lexical variants cited above, but does not replace independent review of all judgments.

### Part2-3.c - Limitations and proposed improvements

| Aspect | Current limitation | Possible improvement and how to assess it |
| --- | --- | --- |
| Retrieval precision | AND checks words in any field without verifying that they describe the requested garment. Advertising text can cause false positives. | Apply garment type, gender and structured attribute constraints; compare P@10 and NDCG@10 using the same human judgments. |
| Coverage | AND misses products because of missing attributes, errors or variants such as polycotton versus cotton. | Normalise synonyms and variants before AND while preserving the required conjunction. Measure recall on an independently judged set. |
| Ranking quality | TF-IDF adds fields with equal weight and does not use proximity, although positions are stored. | Test title and attribute weights, phrase signals and BM25; tune on development data and evaluate on separate queries. |
| Field types | Price, discount, stock and quality_score are stored but do not affect retrieval or ranking. | Add explicit numeric filters and a stock policy; separately evaluate any combination of relevance and rating. |
| Query formulation | Intent, price ranges, spelling errors and ambiguity are not interpreted. | Separate filters from text, suggest corrections and normalise vocabulary without silently relaxing AND. |
| Indexing strategy | Title and description share an index; brands and subcategories have no dedicated indexes. Weights are rebuilt on each run. | Keep fields distinct, add useful attributes and persist indexes; measure quality and runtime before increasing complexity. |
| Own judgments | Rules can be wrong for contradictory attributes, compound garment types or incomplete data. A 0 includes unconfirmed cases. | Have two judges review doubtful cases, allow uncertainty during annotation and agree on final binary labels. |
| Evaluation validity | There are only five own queries, and some match or resemble official queries. The official judged set is small. | Expand independent queries and judgments, separate development and test data, and avoid tuning on the evaluation set. |

These improvements are proposals; the search engine is unchanged in this section to preserve the experimental baseline. The own labels are a rule-based reference, rather than an independent measure of human relevance.